# RealWaste Progres 4 — Controlled Training & Validation Evaluation Runner
Menarik kode modular `src/` dari GitHub, membangun cache `uint8` 128x128, menjalankan **EXP-01 s.d. EXP-04** pada GPU Kaggle, serta menghasilkan artefak visual BAB IV (**khusus Validation Set**, Test Set dikunci untuk Progres 5).

In [ ]:
import os, sys, platform, torch
print('Python:', platform.python_version())
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))


In [ ]:
REPO_URL = 'https://github.com/tryaannn/klasifikasi-jenis-sampah.git'
BRANCH = 'test/majid'
REPO_DIR = '/kaggle/working/klasifikasi-jenis-sampah'

!rm -rf {REPO_DIR}
!git clone --branch {BRANCH} --depth 1 {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
!git log -n 2 --oneline


In [ ]:
# Verifikasi kontrak model & anti-leakage sebelum training penuh
!python -m pytest -q

from pathlib import Path
from src.dataset import resolve_realwaste_root
rw_root = resolve_realwaste_root('/kaggle/input')
print('Resolved RealWaste root:', rw_root)


In [ ]:
CACHE_PATH = '/kaggle/working/realwaste_128_uint8.npz'
OUT_RESULTS = '/kaggle/working/progress4_results'
OUT_FIGURES = '/kaggle/working/progress4_figures'

!python -m src.dataset --data-dir "{rw_root}" --cache-out "{CACHE_PATH}"


In [ ]:
# Eksekusi 4 varian eksperimen terkontrol (EXP-01 .. EXP-04)
!python -m src.train --experiments all \
    --cache-path "{CACHE_PATH}" \
    --output-dir "{OUT_RESULTS}" \
    --patience 8 --seed 42


In [ ]:
# Evaluasi checkpoint terbaik KHUSUS pada Validation Set (713 citra) & buat Gambar 4.1..4.4
!python -m src.evaluate_val \
    --results-dir "{OUT_RESULTS}" \
    --figures-dir "{OUT_FIGURES}" \
    --cache-path "{CACHE_PATH}"


In [ ]:
import json, os
from pathlib import Path
%cd /kaggle/working
# Hapus cache sementara dan clone repo agar kaggle kernels output hanya menarik hasil bersih
if os.path.exists(CACHE_PATH):
    os.remove(CACHE_PATH)
!rm -rf /kaggle/working/klasifikasi-jenis-sampah

comp = json.loads(Path(f'{OUT_RESULTS}/experiment_comparison.json').read_text())
print(json.dumps(comp, indent=2))
